# Learnings 01: Tools And Terminology

This notebook explains the main tools, systems, and technical words used in this project in simple language.

The goal is not just to memorize names. The goal is to understand what each tool does, why we used it, and how it fits into the overall RAG and agentic-RAG system.

By the end of this notebook, we should be comfortable with the vocabulary of the project and the responsibilities of each major component.

## 1. Very Important Basic Terms

### `localhost`
`localhost` means "this same machine". If we open `http://localhost:8000`, we are asking for port `8000` on the current computer.

If our browser is running on a different machine than the server, then `localhost` in the browser means the browser's machine, not the remote server.

### Port
A port is like a numbered door on a machine. Many services can run on the same machine, and ports keep them separate.

Examples from this project:
- API on `8000`
- Airflow on `8081`
- OpenSearch on `9202`
- Langfuse on `3002`

### Hostname
A hostname is the network name of a machine or service. Inside Docker Compose, services talk to each other using service names such as `postgres`, `opensearch`, `redis`, `ollama`, and `langfuse-web`.

### API Key vs Token
An API key is a secret string used to access a service like Jina or Langfuse.
A token is also a secret string, but it is commonly used for authentication in systems like Telegram bots.

### Environment Variable
An environment variable is a configuration value passed to an application at runtime. In this project we store many such values in `.env`.

Examples:
- `JINA_API_KEY`
- `TELEGRAM__BOT_TOKEN`
- `LANGFUSE__PUBLIC_KEY`
- `OPENSEARCH__HOST`

### Health Check
A health check is a small test used by Docker to decide whether a service is alive and responding correctly.

### Worker
A worker is a separate running process used to do work. We saw this idea in multiple places:
- `uvicorn --workers 4` for the API
- Airflow workers / scheduler logic
- Langfuse worker service

## 2. Docker And Docker Compose

### What is Docker?
Docker is a tool used to package an application and its dependencies into a container. A container is a lightweight, isolated runtime environment.

Why Docker mattered in this project:
- we needed many services together
- each service needed its own dependencies
- Docker made startup more repeatable
- it reduced "works on one machine, fails on another" problems

### Image
A Docker image is like a template or blueprint for a container.

### Container
A container is a running instance of an image.

Examples in this project:
- `rag-api`
- `rag-airflow`
- `rag-postgres`
- `rag-opensearch`
- `rag-ollama`
- `rag-telegram-bot`

### Volume
A volume stores data outside the container's temporary filesystem, so data survives restarts.

Examples:
- Postgres data volume
- OpenSearch data volume
- Ollama model volume

### Docker Compose
Docker Compose lets us define and start many related containers together using one `compose.yml` file.

We used it to manage the full stack:
- API
- Airflow
- Postgres
- Redis
- OpenSearch
- Ollama
- Langfuse services
- Telegram bot

### Why Compose was essential
Without Compose, starting the system manually would be painful and error-prone. Compose gives us a reproducible project environment.

## 3. Application Framework, APIs, Async, And UI Tools

### FastAPI
FastAPI is the main backend web framework used in this project. A backend framework helps us expose Python functionality over HTTP so other clients can send requests and receive responses.

In simple words, FastAPI is the main door through which users or client applications talk to our system.

Why FastAPI mattered here:
- it exposes the RAG system as a web service
- it lets us define endpoints like `/ask` and `/ask-agentic`
- it supports async programming well
- it works nicely with typed request and response models

### REST Endpoint
A REST endpoint is a web address that performs a specific backend action.

Examples from this project:
- `/health` checks whether the system is alive
- `/hybrid-search` runs retrieval
- `/ask` runs normal RAG
- `/stream` streams an answer piece by piece
- `/ask-agentic` runs the graph-based agentic pipeline
- `/feedback` is used for feedback and evaluation-related workflows

You can think of endpoints as clearly defined service entry points.

### Request And Response
A request is what the client sends to the server. A response is what the server sends back.

Example:
- request: a question plus settings like `top_k`, `model`, and `use_hybrid`
- response: answer text, sources, chunks used, and other metadata

### APIRouter
An `APIRouter` in FastAPI is a way to organize endpoints into logical groups.

Why this matters:
- it keeps the code cleaner
- it separates features by responsibility
- it makes large APIs easier to manage

In this project we used separate routers for health, search, ask, and agentic ask flows.

### Response Model
A response model defines the shape of the JSON returned by an endpoint.

This matters because:
- it makes the API more predictable
- it helps validation
- it makes the API easier for clients to use correctly

### Async Support
Async support means the system can wait for slow operations like network calls without blocking everything else.

This is important when working with:
- HTTP requests to external services
- model generation
- embeddings APIs
- streaming responses

A simple intuition:
- synchronous style waits in a blocking way
- asynchronous style allows better concurrency for I/O-heavy operations

In this project, async support was already used in several important places such as:
- FastAPI route handlers
- the arXiv client
- the Jina embeddings client
- the Ollama client
- the streaming answer endpoint

### Streaming Response
A streaming response sends output gradually instead of waiting for the full answer to be completed first.

That is why `/stream` can return answer chunks step by step.

### Gradio
Gradio is a Python tool for quickly creating a simple interactive web interface for machine learning or LLM applications.

Why Gradio matters here:
- it provides a lightweight demo UI
- it is easier for experimentation than raw curl commands
- it can help demonstrate the project to others visually

Important project-specific note:
- Gradio code already exists in this project
- but FastAPI is the main backend interface we stabilized and tested most deeply
- Gradio is better thought of as a client or demo interface on top of the backend


## 4. Core Data And Search Tools

### PostgreSQL
PostgreSQL is a relational database. We used it to store structured paper metadata and parsed text.

What we stored there:
- arXiv ID
- title
- authors
- abstract
- parsed raw text
- parser metadata
- processing flags

### OpenSearch
OpenSearch is the search engine for this project. It stores searchable chunks and supports text search plus vector search.

What OpenSearch gave us:
- keyword search
- BM25 ranking
- vector similarity search
- hybrid search
- result highlighting and ranking

### BM25
BM25 is a classical keyword-ranking algorithm. It scores documents based on matching query words, word rarity, and frequency.

Why it matters:
- strong for exact words
- easy to understand
- useful baseline even when vector search exists

### Vector Embeddings
Embeddings convert text into numeric vectors. Similar meanings become close in vector space.

This helps retrieve relevant passages even when exact words differ.

### Jina
Jina AI was used here as the embeddings provider. We sent chunks and queries to Jina to get 1024-dimensional vectors.

Why Jina mattered:
- powers hybrid search
- improves semantic retrieval
- helps when user wording differs from paper wording

### Hybrid Search
Hybrid search combines BM25 and vector search. This gives both exact keyword strength and semantic understanding.

In this project, hybrid search became the main retrieval method because it performed better than keyword-only search for many research questions.

## 5. Parsing, LLM, Orchestration, And Monitoring Tools

### Docling
Docling is a PDF parsing tool. Its job is to read PDFs and extract structured text.

Why it mattered:
- papers come as PDFs
- RAG cannot use PDFs directly
- we needed text extraction before chunking and indexing

We also learned that parsing PDFs is not always simple:
- some PDFs are too large
- some dependencies are missing in containers
- some cached downloads are corrupted

### Ollama
Ollama is a local model-serving system. It lets us run models like `llama3.2:1b` and `qwen2.5:7b` locally or on our server.

Why Ollama mattered:
- answer generation for `/ask`
- step-by-step reasoning for `/ask-agentic`
- local controllable LLM serving

### Airflow
Airflow is a workflow orchestration tool. It is used to run pipelines in organized steps.

In this project it helped us structure the ingestion pipeline:
- fetch metadata
- download PDFs
- parse PDFs
- store papers
- index chunks

### Langfuse
Langfuse is an observability and tracing tool for LLM applications.

Why it matters:
- helps inspect prompts
- helps inspect retrieval and model outputs
- helps compare normal RAG and agentic RAG
- helps debug quality issues later

### Redis
Redis is an in-memory key-value store. In this project, we used it mainly for exact-match caching.

Why caching mattered:
- repeated identical questions do not need full regeneration
- faster responses
- lower repeated model cost/work

### Telegram Bot
The Telegram bot gives a chat interface for asking questions without directly using curl or the API docs.

We learned one important deployment lesson here:
- polling bots should run in a single dedicated process
- they should not be started once per API worker

## 6. Application Terms And RAG Terms

### RAG
RAG stands for Retrieval-Augmented Generation.

Meaning:
1. retrieve relevant information
2. give that information to the LLM
3. generate an answer grounded in those sources

### Chunking
Chunking means splitting a long document into smaller pieces. We do this because search and embeddings work better on smaller meaningful units than on one huge paper body.

### Citation / Source Grounding
Source grounding means the answer should be tied to actual retrieved documents, not invented facts.

### Agentic RAG
Agentic RAG extends normal RAG by adding decision-making steps.

Examples of agentic behavior:
- decide whether retrieval is needed
- retrieve documents
- grade relevance
- rewrite the query if needed
- generate the final answer

### Guardrail
A guardrail is a protective logic step that checks whether a query is in scope or safe before the rest of the workflow continues.

### Trace ID
A trace ID is the identifier used to connect one user request to its logs/steps in Langfuse.

## 7. How The Tools Fit Together In This Project

A simple mental model:

- Docker / Compose: runs the whole ecosystem
- FastAPI: exposes the backend through REST endpoints
- Gradio: offers a lightweight UI client on top of the backend
- Airflow: orchestrates ingestion and indexing steps
- arXiv client: fetches metadata and PDFs
- Docling / parser stack: converts PDFs into usable text
- PostgreSQL: stores paper records and parsed content
- chunker: splits text into searchable chunks
- Jina: creates vectors
- OpenSearch: stores and retrieves chunks
- Ollama: generates answers
- Redis: caches repeated answers
- Langfuse: traces what happened
- Telegram bot: user-facing chat entry point

This is the vocabulary foundation for understanding the next notebook, where we describe the complete end-to-end workflow.